# Reranking Optimization

**Module:** 03 — Reranking

Make reranking affordable: candidate depth, cascades, caching, distillation, batching, and hardware choices without gutting quality.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- List the main cost/latency levers for rerankers
- Design a cascade (cheap filter → strong CE)
- Apply caching keys safely for head queries
- Trade N, batch size, and model size deliberately
- Set SLOs and load-test the rerank stage


## Optimization Levers

**Definition.** **Optimization levers** for reranking include: smaller models, fewer candidates (N), cascades, quantization, distillation, batching, caching, and async prefetch.

**Why it matters.** CE cost scales roughly with N × tokens × model size × QPS. Unoptimized rerank dominates RAG latency and spend.

**How it works.** Measure a baseline → change one lever → re-check nDCG@k and p95 → keep or revert.

**Intuition.** Turn the faucet (N) before replacing the pipes (model)—but measure both.

**Common pitfalls.**
- Cutting N below recall headroom
- Caching without tenant/ACL in the key
- Quantizing into a quality cliff unnoticed
- Micro-optimizing CE when LLM decode is 90% of latency

**When to use.** When rerank p95 or $ exceeds budget but quality still needs CE.

```mermaid
flowchart LR
  L1[Reduce N] --> L2[Cascade]
  L2 --> L3[Cache head]
  L3 --> L4[Smaller CE / distill]
  L4 --> L5[Quantize / batch / GPU]
```


In [ ]:
# Demo 1 — lever dashboard
levers = {
    "N": 50,
    "model": "mini-ce",
    "batch": 16,
    "cascade_threshold": 0.2,
    "cache": True,
}
print(levers)


In [ ]:
# Demo 2 — cost model
def monthly_cost(qps, n, price_per_1k_pairs, hours=24*30):
    pairs = qps * n * 3600 * hours
    return pairs / 1000 * price_per_1k_pairs

print("$/mo", round(monthly_cost(5, 40, 0.5), 2))
print("$/mo", round(monthly_cost(5, 80, 0.5), 2))


In [ ]:
# Demo 3 — cascade filter
def cascade(scores_cheap, strong_score_fn, keep=20):
    idx = sorted(range(len(scores_cheap)), key=lambda i: scores_cheap[i], reverse=True)[:keep]
    strong = [(i, strong_score_fn(i)) for i in idx]
    strong.sort(key=lambda t: t[1], reverse=True)
    return strong

cheap = [0.1, 0.9, 0.4, 0.8, 0.2]
print(cascade(cheap, lambda i: {0:0.2,1:0.5,2:0.9,3:0.7,4:0.1}[i], keep=3))


In [ ]:
# Demo 4 — N sweep vs proxy quality
for n in [10, 20, 40, 80, 160]:
    recall = min(1.0, n / 25)
    latency = 5 + n * 2
    print(f"N={n:3d} recall_proxy={recall:.2f} latency_ms≈{latency}")


### Try it yourself — Optimization Levers

1. Identify which lever you'd pull first if p95 is failing.
2. If gold median rank is 12, argue for a starting N.


## Caching strategies

**Definition.** **Caching** stores rerank scores or final orders for repeated (query, doc-set) keys—especially head queries and FAQ traffic.

**Why it matters.** Head queries can be a large fraction of QPS; CE compute is wasted if recomputed identically.

**How it works.** Key by tenant + normalized query + doc ID version/hash; TTL or invalidate on index update.

**Intuition.** Don't regrade the same essay every hour.

**Common pitfalls.**
- Cache keys missing tenant → cross-tenant leakage
- Never invalidating after corpus updates
- Caching raw prompts with secrets

**When to use.** High-repeat query distributions (FAQ, status pages, support macros).

### Hardware notes

| Setup | Fit |
|-------|-----|
| CPU MiniLM | Low QPS / small N |
| Single GPU | Mid QPS batches |
| API rerank | Little MLOps |
| Distilled tiny CE | Edge / strict latency |


In [ ]:
# Demo 1 — safe cache key
import hashlib

def cache_key(tenant: str, query: str, doc_ids: list[str], model: str) -> str:
    basis = "|".join([tenant, query.strip().lower(), model, ",".join(doc_ids)])
    return hashlib.sha256(basis.encode()).hexdigest()[:16]

print(cache_key("acme", "Refund Window", ["c1", "c2"], "bge-reranker"))


In [ ]:
# Demo 2 — hit rate sketch
requests, unique = 10_000, 2_500
print("upper_bound_hit_rate", 1 - unique/requests)


In [ ]:
# Demo 3 — invalidation
index_version = 3
print("namespace", f"v{index_version}")
print("bump version on reindex to flush stale ranks")


### Try it yourself — Caching strategies

1. Write the cache key fields for your app.
2. When must you bypass cache?


## Distillation, quantization, batching

**Definition.** Teach a small student CE from a large teacher; quantize weights; batch pairs to saturate hardware.

**Why it matters.** These keep quality close while cutting $ and ms—especially at scale.

**How it works.** Collect teacher scores → train student → compare nDCG → deploy behind the same API.

**Intuition.** A senior grades a sample; a junior learns to mimic the ranking cheaply.

**Common pitfalls.**
- Distilling without hard negatives from your domain
- Batch sizes that OOM on long docs
- INT4 cliffs on ranking logits

**When to use.** After N/cascade/cache are tuned and CE still dominates cost.


In [ ]:
# Demo 1 — teacher soft labels
import numpy as np
teacher = np.array([2.0, 0.1, 1.2])
soft = np.exp(teacher); soft /= soft.sum()
print(soft.round(3))


In [ ]:
# Demo 2 — batch packing efficiency
pair_lens = [40, 120, 80, 200]
max_len = max(pair_lens)
waste = sum(max_len - x for x in pair_lens) / (max_len * len(pair_lens))
print("pad_waste", round(waste, 3))


In [ ]:
# Demo 3 — SLO check
p95_ms, slo = 180, 200
print("ok" if p95_ms <= slo else "breach")


### Try it yourself — Distillation, quantization, batching

1. Outline a distill dataset for your top 500 queries.
2. Pick a batch size heuristic for GPU memory you have.


## Glossary

- **cascade**: Cheap stage reduces CE inputs
- **distillation**: Small model mimics large reranker
- **cache key**: Composite identity for memoized scores


### Workshop drill — Reranking Optimization (1)

Restate each major section heading as one exam-ready sentence.


In [ ]:
# Workshop drill 1 — Reranking Optimization
headings = ['Optimization Levers', 'Caching strategies', 'Distillation, quantization, batching']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Reranking Optimization (2)

Sketch a latency budget: first-stage ms + rerank (N candidates × cost) + LLM.


In [ ]:
# Workshop drill 2 — Reranking Optimization
first_ms, per_pair_ms, n, llm_ms = 40, 3, 50, 800
print('total_ms', first_ms + n*per_pair_ms + llm_ms)
print('rerank_share', round(n*per_pair_ms/(first_ms+n*per_pair_ms+llm_ms), 3))


### Workshop drill — Reranking Optimization (3)

Design an offline metric slice: 5 queries with graded relevance labels.


In [ ]:
# Workshop drill 3 — Reranking Optimization
eval_set = [{'q':'...','docs':{'d1':2,'d2':1,'d3':0}}]
print('n_queries', len(eval_set))
print('TODO: fill real labels')


### Workshop drill — Reranking Optimization (4)

Write a go/no-go checklist for shipping a reranker in RAG.


In [ ]:
# Workshop drill 4 — Reranking Optimization
for c in ['latency_p95','nDCG@10','cost/1k','cache_hit','fallback']:
    print(f'[ ] {c}')


### Workshop drill — Reranking Optimization (5)

Compare bi-encoder vs cross-encoder in a small table (fill TODOs).


In [ ]:
# Workshop drill 5 — Reranking Optimization
print('| axis | bi | cross |')
print('|------|----|-------|')
print('| latency | TODO | TODO |')
print('| precision | TODO | TODO |')


## Summary & Key Takeaways

- Tune N and cascades before exotic compression
- Cache with tenant + model + doc versions in the key
- Distill/quantize after measuring real CE dominance
- Guard every change with nDCG and p95 gates

### Practice

Produce a one-page cost model for your expected QPS and N.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
